<a href="https://colab.research.google.com/github/Nayab-khalid/FlyRank-AI-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nayab-khalid/FlyRank-AI-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [2]:
# ============================================================
# ML-09 — PART 1
# TWO PAPER FINDINGS + MY METHODOLOGY QUESTIONS
# ============================================================

# FINDING 1 — THE ANATOMY OF GROWING CONTENT
#
# The paper reports that content with rising impressions tends to
# be longer, younger, and slightly better positioned than content
# with falling impressions.
#
# The growing group has an average of about 3,180 words and
# 184 days of age, while the declining group has about 2,311 words
# and 230 days of age.
#
# The paper reports large groups of approximately 74,187 rising
# pages and 45,272 falling pages.
#
# The paper itself describes this as an observational comparison,
# so the result shows an association rather than proof that being
# younger or longer causes growth.
#
# METHODOLOGY QUESTION:
# Where exactly does the outcome label come from, and is the
# validation design strong enough to support the claim?
#
# In this finding, the groups are defined using rising versus
# falling impressions. The paper's metric description says trend
# direction is calculated from the change in impressions between
# the most recent 30 days and the previous 30 days.
#
# My question is whether the observed differences in age and
# word count could partly reflect differences in content history,
# client mix, or other confounding factors rather than the
# characteristics themselves causing better performance.
#
# A stronger validation design for a causal interpretation would
# compare similar pages or use a time-aware before-and-after design
# while controlling for important differences between pages.
#
# Therefore, I would treat this finding as measured and directional
# evidence about the observed portfolio, not as proof that increasing
# word count or reducing content age will automatically cause growth.


# FINDING 4 — THE FRESHNESS MULTIPLIER
#
# The paper reports that the 31-90 day freshness window is the
# strongest stable freshness band, with a growth-to-decline ratio
# of about 7.88:1.
#
# It also reports that 365+ day content refreshed within 30 days
# showed a 3.2x health increase, from 10.7 to 34.5, and 57x more
# impressions, from 71 to 4,039.
#
# The paper also warns that the 361+ freshness bucket is very
# unstable because it contains only one declining page in the
# local active-content sample.
#
# METHODOLOGY QUESTION:
# How is the refresh exposure defined relative to the outcome
# window, and does the validation design separate association
# from the effect of refreshing?
#
# The important question is whether pages that were refreshed were
# comparable to pages that were not refreshed before the outcome
# was measured. Older pages selected for refresh may already differ
# in demand, importance, quality, or historical performance.
#
# I would therefore want to know whether the comparison used a
# matched or otherwise controlled comparison group, and whether
# the impression and health measurements were taken after the
# refresh in a clearly separated outcome window.
#
# Without that stronger design, the result supports an observed
# association between recent refresh activity and stronger
# performance, but it does not by itself prove that the refresh
# caused the improvement.
#
# Overall, this is especially relevant to my own model because
# I should avoid turning freshness into a causal claim. I should
# use it as a measurable signal for decision-support and validate
# its usefulness with a leakage-safe, client-grouped or time-aware
# evaluation.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [3]:
# ============================================================
# ML-09 — PART 2
# MY MODEL UNDER AN HONEST SPLIT
# ============================================================

# REASONING:
# My Week-5 model used a client-grouped split, which is a more
# honest test when observations from the same client can share
# characteristics.
#
# This audit compares a simple row-based split with the grouped
# client split so that I can see whether performance changes when
# client overlap is removed.
#
# The comparison uses the same Logistic Regression method and the
# same Average Precision metric.
#
# The grouped result is treated as the more honest estimate of
# performance for unseen clients.

%pip install -q pandas numpy scikit-learn

import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------

possible_paths = [
    "data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "/content/data/raw/content_refresh_anonymized.csv"
]

csv_path = next(
    (p for p in possible_paths if os.path.exists(p)),
    None
)

if csv_path is None:
    csv_url = (
        "https://raw.githubusercontent.com/"
        "Nayab-khalid/FlyRank-AI-Internship/"
        "main/data/raw/content_refresh_anonymized.csv"
    )
    df = pd.read_csv(csv_url)
else:
    df = pd.read_csv(csv_path)

# ------------------------------------------------------------
# LABEL
# ------------------------------------------------------------

df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

# ------------------------------------------------------------
# FEATURES
# ------------------------------------------------------------

numeric_features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

categorical_features = [
    "content_type",
    "main_intent",
    "competition_level"
]

numeric_features = [
    c for c in numeric_features if c in df.columns
]

categorical_features = [
    c for c in categorical_features if c in df.columns
]

features = numeric_features + categorical_features

X = df[features]
y = df["is_declining_label"]
groups = df["client_id"]

# ------------------------------------------------------------
# MODEL PIPELINE
# ------------------------------------------------------------

preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        numeric_features
    ),
    (
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ))
        ]),
        categorical_features
    )
])

def make_model():
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ])

# ------------------------------------------------------------
# BEFORE — ROW-BASED SPLIT
# ------------------------------------------------------------

X_train_old, X_test_old, y_train_old, y_test_old = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

old_model = make_model()
old_model.fit(X_train_old, y_train_old)

old_probability = old_model.predict_proba(
    X_test_old
)[:, 1]

old_ap = average_precision_score(
    y_test_old,
    old_probability
)

# ------------------------------------------------------------
# AFTER — CLIENT-GROUPED SPLIT
# ------------------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train_new = X.iloc[train_idx]
X_test_new = X.iloc[test_idx]

y_train_new = y.iloc[train_idx]
y_test_new = y.iloc[test_idx]

group_model = make_model()
group_model.fit(X_train_new, y_train_new)

group_probability = group_model.predict_proba(
    X_test_new
)[:, 1]

group_ap = average_precision_score(
    y_test_new,
    group_probability
)

# ------------------------------------------------------------
# BEFORE / AFTER TABLE
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "validation_design": [
        "Row-based split",
        "Client-grouped split"
    ],
    "average_precision": [
        old_ap,
        group_ap
    ]
})

print("=" * 60)
print("BEFORE / AFTER VALIDATION")
print("=" * 60)

display(comparison)

print(
    "\nThe client-grouped result is the more conservative estimate "
    "because test clients are not present in training."
)


BEFORE / AFTER VALIDATION


,validation_design,average_precision
0,Row-based split,0.935525
1,Client-grouped split,0.871540



The client-grouped result is the more conservative estimate because test clients are not present in training.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [4]:
# ============================================================
# ML-09 — PART 3
# LEAKAGE AUDIT
# ============================================================

# REASONING:
# I audit the final feature list for fields that directly define
# the label or contain outcome information.
#
# trend_direction is the source of the decline label and must not
# be used as a feature.
#
# trend_pct is derived from the same trend outcome and must also
# be excluded.
#
# is_declining_label is the target itself and must never be used
# as an input feature.
#
# Future-window variables must also be excluded if they contain
# information from the period being predicted.
#
# Client and content identifiers are grouping or joining fields,
# not predictive features.

# ------------------------------------------------------------
# FINAL FEATURE LIST
# ------------------------------------------------------------

final_features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "content_type",
    "main_intent",
    "competition_level"
]

forbidden = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "client_id",
    "content_id"
]

leakage_found = [
    feature
    for feature in final_features
    if feature in forbidden
]

print("=" * 60)
print("LEAKAGE AUDIT")
print("=" * 60)

print("Number of forbidden fields in final feature list:",
      len(leakage_found))

if len(leakage_found) == 0:
    print("✓ No explicitly forbidden label/ID fields found.")
else:
    print("⚠ Potential leakage fields:")
    print(leakage_found)

# ------------------------------------------------------------
# EXPLICIT LABEL CHECK
# ------------------------------------------------------------

if "is_declining_label" not in final_features:
    print("✓ Target is not used as a feature.")

if "trend_direction" not in final_features:
    print("✓ trend_direction is not used as a feature.")

if "trend_pct" not in final_features:
    print("✓ trend_pct is not used as a feature.")

print("""
LEAKAGE CONCLUSION:

The audited feature list does not contain the target or the
documented label-derived trend fields. Identifier fields are also
not used as predictive inputs. Any future-window variable must be
excluded if it overlaps the outcome period.
""")


LEAKAGE AUDIT
Number of forbidden fields in final feature list: 0
✓ No explicitly forbidden label/ID fields found.
✓ Target is not used as a feature.
✓ trend_direction is not used as a feature.
✓ trend_pct is not used as a feature.

LEAKAGE CONCLUSION:

The audited feature list does not contain the target or the
documented label-derived trend fields. Identifier fields are also
not used as predictive inputs. Any future-window variable must be
excluded if it overlaps the outcome period.



## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [5]:
# ============================================================
# ML-09 — PART 4
# CLAIM REWRITE
# ============================================================

# ORIGINAL BOLD CLAIM:
# "The Logistic Regression model predicts which pages will decline
# and can identify pages that Google will rank lower."

# SAFE REWRITE:
# "In this dataset, the Logistic Regression model showed measured
# and directional ability to rank pages associated with the
# observed decline label. Its performance should be interpreted
# as decision-support evidence rather than a prediction of
# Google's ranking algorithm or a causal claim about why a page
# declined."

print("""
ORIGINAL CLAIM:
The Logistic Regression model predicts which pages will decline
and can identify pages that Google will rank lower.

REWRITTEN CLAIM:
In this dataset, the Logistic Regression model showed measured
and directional ability to rank pages associated with the observed
decline label. Its performance should be interpreted as
decision-support evidence rather than a prediction of Google's
ranking algorithm or a causal claim about why a page declined.
""")



ORIGINAL CLAIM:
The Logistic Regression model predicts which pages will decline
and can identify pages that Google will rank lower.

REWRITTEN CLAIM:
In this dataset, the Logistic Regression model showed measured
and directional ability to rank pages associated with the observed
decline label. Its performance should be interpreted as
decision-support evidence rather than a prediction of Google's
ranking algorithm or a causal claim about why a page declined.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.